In [1]:
import pandas as pd
import os
from datetime import datetime, timedelta

directory = 'C:/Users/tuan-/Downloads/1 Lobster Thesis/Data/SPY2023/'

# Konverter tid
def convert_to_datetime(seconds, base_date):
    base_time = datetime.strptime(base_date, '%Y-%m-%d')
    return base_time + timedelta(seconds=seconds)

# monthly data
monthly_data = []

year = 2023

 # Loop monthly
for month in range(1, 13): 
    monthly_files = []
    
    for filename in os.listdir(directory):
        if f'{year}-{month:02}' in filename and 'message' in filename:  # Match filer monthly and year
            message_file = os.path.join(directory, filename)
            orderbook_file = message_file.replace('message', 'orderbook')  # Match orderbook fil
            
            # Gå i stå, slet tomme filer
            if os.path.getsize(message_file) == 0 or os.path.getsize(orderbook_file) == 0:
                print(f"Skipping empty file: {message_file} or {orderbook_file}")
                continue  

            try:
        
                message_df = pd.read_csv(message_file, encoding='utf-8', low_memory=False)
                orderbook_df = pd.read_csv(orderbook_file, encoding='utf-8', low_memory=False)
            except Exception as e:
                print(f"Error loading file {filename}: {e}")
                continue  # Skip this file if there's an error

            # Dropper col 7
            message_df = message_df.iloc[:, :-1]  # Drop last column
            message_df.columns = ['Time (sec)', 'Event Type', 'Order ID', 'Size', 'Price', 'Direction']
            orderbook_df.columns = ['Ask Price 1', 'Ask Size 1', 'Bid Price 1', 'Bid Size 1', 
                                    'Ask Price 2', 'Ask Size 2', 'Bid Price 2', 'Bid Size 2']

            # Start dag SPY_2023-01-03
            base_date = filename.split('_')[1]
            
            
            message_df['Time (sec)'] = message_df['Time (sec)'].apply(lambda x: convert_to_datetime(x, base_date))

            # Merge message and orderbook 
            combined_df = pd.merge(orderbook_df, message_df, left_index=True, right_index=True, how='left')

            # Slet NAN
            combined_df.dropna(inplace=True)

            # Set 'Time (sec)' as index for 1 sec
            combined_df.set_index('Time (sec)', inplace=True)

            # Sampler data 1 sec interval første obs hvert sec
            resampled_df = combined_df.resample('1S').first()

            
            monthly_files.append(resampled_df)
    
    # Concatenate all daily files for the month
    if monthly_files:
        monthly_data_df = pd.concat(monthly_files)
        monthly_data.append(monthly_data_df)
        
        
        monthly_data_df.to_csv(f'processed_{year}_{month:02}.csv', index=False)


final_df = pd.concat(monthly_data)


final_df.to_csv(f'combined_SPY_{year}_cleaned.csv', index=False)

print(final_df.head())

                     Ask Price 1  Ask Size 1  Bid Price 1  Bid Size 1  \
Time (sec)                                                              
2023-01-03 09:30:00    3844100.0       100.0    3843400.0       300.0   
2023-01-03 09:30:01    3844200.0       700.0    3844000.0       200.0   
2023-01-03 09:30:02    3844100.0       400.0    3843400.0       197.0   
2023-01-03 09:30:03    3843800.0       118.0    3843400.0       200.0   
2023-01-03 09:30:04    3843900.0       800.0    3843400.0       300.0   

                     Ask Price 2  Ask Size 2  Bid Price 2  Bid Size 2  \
Time (sec)                                                              
2023-01-03 09:30:00    3844200.0       600.0    3843300.0       600.0   
2023-01-03 09:30:01    3844300.0       800.0    3843900.0       300.0   
2023-01-03 09:30:02    3844200.0       700.0    3843300.0       820.0   
2023-01-03 09:30:03    3843900.0       700.0    3843300.0       420.0   
2023-01-03 09:30:04    3844000.0       700.0    38

In [17]:
print(final_df.tail())

                     Ask Price 1  Ask Size 1  Bid Price 1  Bid Size 1  \
Time (sec)                                                              
2023-12-29 15:59:55    4754400.0       900.0    4754300.0       100.0   
2023-12-29 15:59:56    4754200.0       800.0    4754100.0       300.0   
2023-12-29 15:59:57    4753100.0      1587.0    4753000.0       800.0   
2023-12-29 15:59:58    4753100.0      1200.0    4752900.0       600.0   
2023-12-29 15:59:59    4753100.0       600.0    4752900.0       600.0   

                     Ask Price 2  Ask Size 2  Bid Price 2  Bid Size 2  \
Time (sec)                                                              
2023-12-29 15:59:55    4754500.0       800.0    4754200.0      1670.0   
2023-12-29 15:59:56    4754300.0       800.0    4754000.0       500.0   
2023-12-29 15:59:57    4753200.0      1200.0    4752900.0      1300.0   
2023-12-29 15:59:58    4753200.0      2100.0    4752800.0      1200.0   
2023-12-29 15:59:59    4753200.0      2000.0    47

In [3]:
# Observations (rows) in the final combined DataFrame for the year
total_observations_final = len(final_df)

print(f"Total observations in the final combined DataFrame: {total_observations_final}")

Total observations in the final combined DataFrame: 5740269


In [4]:
final_df.head()

,Ask Price 1,Ask Size 1,Bid Price 1,Bid Size 1,Ask Price 2,Ask Size 2,Bid Price 2,Bid Size 2,Event Type,Order ID,Size,Price,Direction
Time (sec),,,,,,,,,,,,,
2023-01-03 09:30:00,3844100.0,100.0,3843400.0,300.0,3844200.0,600.0,3843300.0,600.0,1.0,39946724.0,100.0,3843400.0,1.0
2023-01-03 09:30:01,3844200.0,700.0,3844000.0,200.0,3844300.0,800.0,3843900.0,300.0,1.0,40482608.0,200.0,3844000.0,1.0
2023-01-03 09:30:02,3844100.0,400.0,3843400.0,197.0,3844200.0,700.0,3843300.0,820.0,4.0,40787052.0,200.0,3844100.0,-1.0
2023-01-03 09:30:03,3843800.0,118.0,3843400.0,200.0,3843900.0,700.0,3843300.0,420.0,3.0,41131364.0,500.0,3843300.0,1.0
2023-01-03 09:30:04,3843900.0,800.0,3843400.0,300.0,3844000.0,700.0,3843300.0,100.0,3.0,41361344.0,500.0,3843300.0,1.0


In [5]:
final_df.tail()

,Ask Price 1,Ask Size 1,Bid Price 1,Bid Size 1,Ask Price 2,Ask Size 2,Bid Price 2,Bid Size 2,Event Type,Order ID,Size,Price,Direction
Time (sec),,,,,,,,,,,,,
2023-12-29 15:59:55,4754400.0,900.0,4754300.0,100.0,4754500.0,800.0,4754200.0,1670.0,3.0,876117912.0,400.0,4754400.0,-1.0
2023-12-29 15:59:56,4754200.0,800.0,4754100.0,300.0,4754300.0,800.0,4754000.0,500.0,3.0,876365300.0,300.0,4754200.0,-1.0
2023-12-29 15:59:57,4753100.0,1587.0,4753000.0,800.0,4753200.0,1200.0,4752900.0,1300.0,3.0,876541100.0,100.0,4753000.0,1.0
2023-12-29 15:59:58,4753100.0,1200.0,4752900.0,600.0,4753200.0,2100.0,4752800.0,1200.0,4.0,876747648.0,200.0,4753000.0,1.0
2023-12-29 15:59:59,4753100.0,600.0,4752900.0,600.0,4753200.0,2000.0,4752800.0,1300.0,4.0,876923912.0,200.0,4753100.0,-1.0


In [6]:
# # Save 
# final_df.to_csv('final_combined_2023.csv', index=True)  # Keep the index to retain 'Time (sec)'

final_df.to_csv('final_combined_2023_with_time.csv', index=True)

In [18]:
# Count unique days
final_df['Date'] = final_df.index.date

# Count the number of unique trading days
unique_trading_days = final_df['Date'].nunique()

print(f"Number of unique trading days: {unique_trading_days}")

# Count the number of observations (rows) in the resampled DataFrame
num_observations = len(resampled_df)

# 
print(f'Total number of observations: {num_observations}')

Number of unique trading days: 249
Total number of observations: 23400


In [8]:
# Load the data from 'final_combined_2023_with_time.csv'
final_combined_2023_with_time = pd.read_csv('final_combined_2023_with_time.csv')


final_combined_2023_with_time['Time (sec)'] = pd.to_datetime(final_combined_2023_with_time['Time (sec)'])

# Group trading days
grouped = final_combined_2023_with_time.groupby(final_combined_2023_with_time['Time (sec)'].dt.date)

# Sampler liste
resampled_5min_list = []

# Iterate through each group each day
for date, group in grouped:
    # Filtrer data for trading hours (9:30 AM to 4:00 PM)
    group_trading_hours = group[
        (group['Time (sec)'].dt.time >= pd.to_datetime('09:30:00').time()) &
        (group['Time (sec)'].dt.time <= pd.to_datetime('16:00:00').time())
    ]
    
    # Resample for 5-minute intervals 
    resampled_day = group_trading_hours.resample('5T', on='Time (sec)').agg({
        'Ask Price 1': ['first', 'max', 'min', 'last'],
        'Bid Price 1': ['first', 'max', 'min', 'last'],
        'Ask Price 2': ['first', 'max', 'min', 'last'],
        'Bid Price 2': ['first', 'max', 'min', 'last'],
        'Ask Size 1': ['sum', 'mean'],
        'Bid Size 1': ['sum', 'mean'],
        'Ask Size 2': ['sum', 'mean'],
        'Bid Size 2': ['sum', 'mean'],
        'Price': ['first', 'max', 'min', 'last'],
        'Direction': 'mean'
    })
    
   
    resampled_day.columns = ['_'.join(col).strip() for col in resampled_day.columns.values]
    
    # Append the resampled data dag
    resampled_5min_list.append(resampled_day)

# Concatenate all, DataFrame
resampled_5min_final_df = pd.concat(resampled_5min_list)

# Reset index 'Time (sec)' 
resampled_5min_final_df.reset_index(inplace=True)

In [9]:
print(resampled_5min_final_df.head(10))

           Time (sec)  Ask Price 1_first  Ask Price 1_max  Ask Price 1_min  \
0 2023-01-03 09:30:00          3844100.0        3851300.0        3836200.0   
1 2023-01-03 09:35:00          3845400.0        3854600.0        3843500.0   
2 2023-01-03 09:40:00          3854100.0        3864000.0        3852100.0   
3 2023-01-03 09:45:00          3852900.0        3853700.0        3830600.0   
4 2023-01-03 09:50:00          3835100.0        3839600.0        3825500.0   
5 2023-01-03 09:55:00          3828300.0        3831000.0        3816900.0   
6 2023-01-03 10:00:00          3817300.0        3825600.0        3817100.0   
7 2023-01-03 10:05:00          3821400.0        3823700.0        3818000.0   
8 2023-01-03 10:10:00          3819900.0        3820500.0        3809500.0   
9 2023-01-03 10:15:00          3811800.0        3813700.0        3802400.0   

   Ask Price 1_last  Bid Price 1_first  Bid Price 1_max  Bid Price 1_min  \
0         3845000.0          3843400.0        3851100.0        38

In [10]:
# Save
resampled_5min_final_df.to_csv('resampled_5min_final_2023_corrected.csv', index=False)

In [11]:
# Count observations for each day in 5-minute interval
resampled_5min_final_df['Date'] = resampled_5min_final_df['Time (sec)'].dt.date

# Count the number of rows for each day
observations_per_day = resampled_5min_final_df.groupby('Date').size()


print(observations_per_day)

Date
2023-01-03    78
2023-01-04    78
2023-01-05    78
2023-01-06    78
2023-01-09    78
              ..
2023-12-22    78
2023-12-26    78
2023-12-27    78
2023-12-28    78
2023-12-29    78
Length: 249, dtype: int64


In [12]:
print(resampled_5min_final_df.head(10))

# Save igen
resampled_5min_final_df.to_csv('resampled_5min_final_2023_corrected.csv', index=False)

           Time (sec)  Ask Price 1_first  Ask Price 1_max  Ask Price 1_min  \
0 2023-01-03 09:30:00          3844100.0        3851300.0        3836200.0   
1 2023-01-03 09:35:00          3845400.0        3854600.0        3843500.0   
2 2023-01-03 09:40:00          3854100.0        3864000.0        3852100.0   
3 2023-01-03 09:45:00          3852900.0        3853700.0        3830600.0   
4 2023-01-03 09:50:00          3835100.0        3839600.0        3825500.0   
5 2023-01-03 09:55:00          3828300.0        3831000.0        3816900.0   
6 2023-01-03 10:00:00          3817300.0        3825600.0        3817100.0   
7 2023-01-03 10:05:00          3821400.0        3823700.0        3818000.0   
8 2023-01-03 10:10:00          3819900.0        3820500.0        3809500.0   
9 2023-01-03 10:15:00          3811800.0        3813700.0        3802400.0   

   Ask Price 1_last  Bid Price 1_first  Bid Price 1_max  Bid Price 1_min  \
0         3845000.0          3843400.0        3851100.0        38

In [13]:
from IPython.display import display, HTML

# Display as an HTML tabel
html_table = resampled_5min_final_df.head(10).to_html(index=False)
display(HTML(html_table))

# Gem denne
resampled_5min_final_df.to_csv('resampled_5min_final_2023_corrected.csv', index=False)

Time (sec),Ask Price 1_first,Ask Price 1_max,Ask Price 1_min,Ask Price 1_last,Bid Price 1_first,Bid Price 1_max,Bid Price 1_min,Bid Price 1_last,Ask Price 2_first,Ask Price 2_max,Ask Price 2_min,Ask Price 2_last,Bid Price 2_first,Bid Price 2_max,Bid Price 2_min,Bid Price 2_last,Ask Size 1_sum,Ask Size 1_mean,Bid Size 1_sum,Bid Size 1_mean,Ask Size 2_sum,Ask Size 2_mean,Bid Size 2_sum,Bid Size 2_mean,Price_first,Price_max,Price_min,Price_last,Direction_mean,Date
2023-01-03 09:30:00,3844100.0,3851300.0,3836200.0,3845000.0,3843400.0,3851100.0,3836000.0,3844600.0,3844200.0,3851500.0,3836300.0,3845100.0,3843300.0,3850800.0,3835900.0,3844400.0,77387.0,257.956667,74873.0,249.576667,128072.0,426.906667,123276.0,410.920000,3843400.0,3850900.0,3836200.0,3844600.0,0.073333,2023-01-03
2023-01-03 09:35:00,3845400.0,3854600.0,3843500.0,3853400.0,3844900.0,3854300.0,3843200.0,3853100.0,3845500.0,3854700.0,3843600.0,3853500.0,3844800.0,3854200.0,3843100.0,3853000.0,123694.0,412.313333,91199.0,303.996667,156271.0,520.903333,167164.0,557.213333,3845400.0,3854600.0,3843500.0,3853100.0,0.026667,2023-01-03
2023-01-03 09:40:00,3854100.0,3864000.0,3852100.0,3852600.0,3853700.0,3863600.0,3851600.0,3852200.0,3854200.0,3864100.0,3852200.0,3852700.0,3853600.0,3863500.0,3851500.0,3852100.0,104270.0,347.566667,134308.0,447.693333,94597.0,315.323333,119734.0,399.113333,3854100.0,3864000.0,3851500.0,3852200.0,0.033333,2023-01-03
2023-01-03 09:45:00,3852900.0,3853700.0,3830600.0,3835200.0,3852600.0,3853300.0,3830400.0,3835100.0,3853200.0,3853800.0,3830700.0,3835300.0,3852500.0,3853200.0,3830300.0,3834900.0,110190.0,367.300000,119914.0,399.713333,142322.0,474.406667,146912.0,489.706667,3852600.0,3853600.0,3830500.0,3835200.0,-0.153333,2023-01-03
2023-01-03 09:50:00,3835100.0,3839600.0,3825500.0,3828300.0,3834900.0,3839300.0,3825100.0,3827900.0,3835200.0,3839700.0,3825600.0,3828400.0,3834700.0,3839200.0,3825000.0,3827800.0,118290.0,394.300000,89197.0,297.323333,143619.0,478.730000,146215.0,487.383333,3835100.0,3839300.0,3825400.0,3828300.0,-0.080000,2023-01-03
2023-01-03 09:55:00,3828300.0,3831000.0,3816900.0,3817200.0,3827900.0,3830800.0,3816600.0,3816700.0,3828400.0,3831100.0,3817000.0,3817300.0,3827800.0,3830700.0,3816500.0,3816600.0,103659.0,345.530000,106175.0,353.916667,157338.0,524.460000,141644.0,472.146667,3827900.0,3831000.0,3816800.0,3817200.0,-0.060000,2023-01-03
2023-01-03 10:00:00,3817300.0,3825600.0,3817100.0,3821300.0,3817000.0,3825500.0,3816900.0,3820900.0,3817400.0,3825700.0,3817200.0,3821400.0,3816900.0,3825400.0,3816800.0,3820800.0,68320.0,227.733333,70727.0,235.756667,97452.0,324.840000,108534.0,361.780000,3817000.0,3825500.0,3817000.0,3820900.0,-0.086667,2023-01-03
2023-01-03 10:05:00,3821400.0,3823700.0,3818000.0,3820400.0,3821000.0,3823500.0,3817700.0,3820100.0,3821500.0,3823800.0,3818100.0,3820500.0,3820900.0,3823400.0,3817600.0,3819900.0,78715.0,262.383333,80660.0,268.866667,109478.0,364.926667,121728.0,405.760000,3821400.0,3823700.0,3817600.0,3819900.0,-0.060000,2023-01-03
2023-01-03 10:10:00,3819900.0,3820500.0,3809500.0,3811500.0,3819500.0,3820100.0,3809300.0,3811200.0,3820000.0,3820600.0,3809600.0,3811600.0,3819400.0,3820000.0,3809200.0,3811100.0,93987.0,313.290000,98936.0,329.786667,166385.0,554.616667,171357.0,571.190000,3819900.0,3820300.0,3809400.0,3811500.0,-0.100000,2023-01-03
2023-01-03 10:15:00,3811800.0,3813700.0,3802400.0,3802700.0,3811600.0,3813500.0,3802200.0,3802500.0,3811900.0,3813800.0,3802500.0,3802800.0,3811500.0,3813400.0,3802100.0,3802400.0,70433.0,234.776667,62063.0,206.876667,79044.0,263.480000,105404.0,351.346667,3811800.0,3813500.0,3802200.0,3802500.0,0.106667,2023-01-03


In [14]:
from IPython.display import display, HTML

# Display as an HTML tabel
html_table = resampled_5min_final_df.tail(10).to_html(index=False)
display(HTML(html_table))

# Gem denne
resampled_5min_final_df.to_csv('resampled_5min_final_2023_corrected.csv', index=False)

Time (sec),Ask Price 1_first,Ask Price 1_max,Ask Price 1_min,Ask Price 1_last,Bid Price 1_first,Bid Price 1_max,Bid Price 1_min,Bid Price 1_last,Ask Price 2_first,Ask Price 2_max,Ask Price 2_min,Ask Price 2_last,Bid Price 2_first,Bid Price 2_max,Bid Price 2_min,Bid Price 2_last,Ask Size 1_sum,Ask Size 1_mean,Bid Size 1_sum,Bid Size 1_mean,Ask Size 2_sum,Ask Size 2_mean,Bid Size 2_sum,Bid Size 2_mean,Price_first,Price_max,Price_min,Price_last,Direction_mean,Date
2023-12-29 15:10:00,4757200.0,4758900.0,4754800.0,4757100.0,4757100.0,4758800.0,4754700.0,4756800.0,4757300.0,4759000.0,4754900.0,4757200.0,4757000.0,4758700.0,4754600.0,4756700.0,165001.0,551.842809,123779.0,413.976589,314120.0,1050.568562,265050.0,886.454849,4757100.0,4758900.0,4754800.0,4757200.0,-0.010033,2023-12-29
2023-12-29 15:15:00,4757100.0,4757800.0,4754500.0,4754900.0,4757000.0,4757700.0,4754400.0,4754800.0,4757200.0,4757900.0,4754600.0,4755000.0,4756900.0,4757600.0,4754300.0,4754700.0,203306.0,677.686667,125092.0,416.973333,331316.0,1104.386667,193577.0,645.256667,4757200.0,4757800.0,4754500.0,4754800.0,-0.006667,2023-12-29
2023-12-29 15:20:00,4754700.0,4755500.0,4751400.0,4754600.0,4754600.0,4755400.0,4751300.0,4754400.0,4754800.0,4755600.0,4751500.0,4754700.0,4754500.0,4755300.0,4751200.0,4754300.0,185373.0,617.910000,139940.0,466.466667,292723.0,975.743333,188403.0,628.010000,4754700.0,4755500.0,4751400.0,4754600.0,-0.020000,2023-12-29
2023-12-29 15:25:00,4754700.0,4758600.0,4754500.0,4757700.0,4754600.0,4758500.0,4754400.0,4757600.0,4754800.0,4758700.0,4754600.0,4757800.0,4754500.0,4758400.0,4754300.0,4757500.0,200272.0,667.573333,143337.0,477.790000,320978.0,1069.926667,258581.0,861.936667,4754700.0,4758500.0,4754400.0,4757600.0,-0.020000,2023-12-29
2023-12-29 15:30:00,4757800.0,4757800.0,4754500.0,4754900.0,4757600.0,4757700.0,4754400.0,4754800.0,4757900.0,4757900.0,4754600.0,4755000.0,4757500.0,4757600.0,4754300.0,4754700.0,275013.0,916.710000,151600.0,505.333333,417146.0,1390.486667,312771.0,1042.570000,4757600.0,4757900.0,4754500.0,4754900.0,-0.073333,2023-12-29
2023-12-29 15:35:00,4754900.0,4756500.0,4754900.0,4756000.0,4754800.0,4756400.0,4754800.0,4755800.0,4755000.0,4756600.0,4755000.0,4756100.0,4754700.0,4756300.0,4754700.0,4755700.0,266983.0,889.943333,171661.0,572.203333,436226.0,1454.086667,279326.0,931.086667,4754900.0,4756500.0,4754800.0,4755800.0,-0.046667,2023-12-29
2023-12-29 15:40:00,4755900.0,4757500.0,4753400.0,4753400.0,4755800.0,4757400.0,4753300.0,4753300.0,4756000.0,4757600.0,4753500.0,4753500.0,4755700.0,4757300.0,4753200.0,4753200.0,237495.0,791.650000,193930.0,646.433333,355560.0,1185.200000,292566.0,975.220000,4756000.0,4757600.0,4753300.0,4753400.0,-0.300000,2023-12-29
2023-12-29 15:45:00,4753200.0,4754900.0,4751900.0,4751900.0,4753100.0,4754800.0,4751700.0,4751700.0,4753300.0,4755000.0,4752000.0,4752000.0,4753000.0,4754700.0,4751600.0,4751600.0,249110.0,830.366667,167396.0,557.986667,392769.0,1309.230000,245308.0,817.693333,4753200.0,4754900.0,4751600.0,4751600.0,-0.160000,2023-12-29
2023-12-29 15:50:00,4752100.0,4756000.0,4747300.0,4747600.0,4752000.0,4755900.0,4747200.0,4747500.0,4752200.0,4756100.0,4747400.0,4747700.0,4751900.0,4755800.0,4747100.0,4747400.0,185772.0,619.240000,251356.0,837.853333,307576.0,1025.253333,431335.0,1437.783333,4752100.0,4756000.0,4747200.0,4747600.0,0.300000,2023-12-29
2023-12-29 15:55:00,4747500.0,4757000.0,4745300.0,4753100.0,4747400.0,4756900.0,4745200.0,4752900.0,4747600.0,4757100.0,4745400.0,4753200.0,4747300.0,4756800.0,4745100.0,4752800.0,208354.0,694.513333,273990.0,913.300000,282233.0,940.776667,449790.0,1499.300000,4747500.0,4756900.0,4745200.0,4753100.0,-0.006667,2023-12-29
